# V2 Phase 7 — Colab GPU smoke (Qwen3-8B)

**Before running:** Runtime → Change runtime type → **GPU**.

## Setup instructions

1. **Push your latest V2 code to GitHub** (commit + push before running Colab).
2. Open this notebook in Colab with **GPU** runtime.
3. In the next cell, check **`REPO_URL`** and **`BRANCH`** match your GitHub repo.
4. Run all cells in order.

The setup cell **clones the repo** into `/content/` — no need to copy `V2/` to Google Drive.

**Optional (after smoke test):** mount Drive in a separate cell to copy `results/config/*.json` to `My Drive/MSc-RAG/` for backup.

**Expected outputs:**
- `results/config/phase7_runtime_fingerprint.json`
- `results/config/phase7_smoke_test.json`

## 1. Clone GitHub repo and enter V2

In [ ]:
from pathlib import Path
import os
import sys

# ========== EDIT IF NEEDED ==========
REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'main'  # e.g. main, master, or your feature branch
CLONE_DIR = Path('/content/capstone-rag')
# ====================================

if CLONE_DIR.exists():
    !rm -rf {CLONE_DIR}

!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}

V2_ROOT = CLONE_DIR / 'V2'
if not V2_ROOT.is_dir():
    raise FileNotFoundError(f'V2/ not found in repo. Check BRANCH has V2/: {V2_ROOT}')
if not (V2_ROOT / 'scripts' / 'smoke_generate.py').is_file():
    raise FileNotFoundError(f'Invalid V2 root: {V2_ROOT}')

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
print('OK — V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Run smoke test (llama_cpp on Colab GPU)

In [ ]:
!PYTHONPATH=. python scripts/smoke_generate.py --backend llama_cpp --notebook notebooks/colab_phase7_smoke.ipynb

In [ ]:
# Fallback only if llama_cpp fails:
# !PYTHONPATH=. python scripts/smoke_generate.py --backend transformers --notebook notebooks/colab_phase7_smoke.ipynb

## 4. Check results

In [ ]:
import json
from pathlib import Path

fp = Path('results/config/phase7_runtime_fingerprint.json')
smoke = Path('results/config/phase7_smoke_test.json')
print('fingerprint:', fp.is_file())
print('smoke_test:', smoke.is_file())
if smoke.is_file():
    data = json.loads(smoke.read_text())
    print('status:', data.get('status'))
    print('actual:', repr(data.get('actual')))

## 5. (Optional) Copy results to Google Drive

In [ ]:
# Run this cell only if you want to back up smoke outputs to Drive.
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

dest = Path('/content/drive/MyDrive/MSc-RAG/configs/phase7')
dest.mkdir(parents=True, exist_ok=True)

for name in ('phase7_runtime_fingerprint.json', 'phase7_smoke_test.json'):
    src = Path('results/config') / name
    if src.is_file():
        shutil.copy2(src, dest / name)
        print('copied', name, '->', dest / name)